# SAM 2 for Roman Space Telescope: Tutorial 02
## Fine-Tuning SAM 2 on Astronomical Datasets (Optimized for NVIDIA A100 40GB)

This tutorial explains **how to fine-tune Meta's Segment Anything Model 2 (SAM 2)** on astronomical data, detailing:
1. **Prerequisites**: What you need to fine-tune SAM 2 (FITS data, ground-truth masks, prompt conditioning).
2. **Data Formats**: Single master FITS tiling vs directory structure, integer labeled segmaps.
3. **Pipeline Architecture**: Vision backbone freezing, prompt derivation, combined BCE + Dice loss, validation metrics.
4. **A100 Resource Scaling**: Scaling batch size to 16–32 on 40GB VRAM, TensorFloat-32 (TF32), native bfloat16 AMP, and multi-worker DataLoader.
5. **Complete, Runnable Walkthrough**: Live training loop, convergence tracking, checkpoint saving, and post-training inference with default `leakySB` stretch.

### Google Colab Setup (A100 GPU with 40GB VRAM)

To configure your Google Colab instance for maximum throughput:
1. Go to **Runtime -> Change runtime type**.
2. Under **Hardware accelerator**, select **A100 GPU**.
3. Under **Runtime shape**, select **High-RAM** (giving 12 vCPUs and ~83.5 GB system memory).
4. Run the installation cell below.

In [ ]:
# In Google Colab, uncomment and run:
# !nvidia-smi
# !git clone https://github.com/PabloMSanAla/sam2roman.git
# %cd sam2roman
# !pip install -e .

### Step 1: Import Libraries and Detect Accelerator Profile

In [ ]:
import os
import sys
from pathlib import Path

for p in ['/content/sam2roman/src', 'src', '../src', '../../src']:
    resolved = Path(p).resolve()
    if resolved.exists() and str(resolved) not in sys.path:
        sys.path.insert(0, str(resolved))
        break

import matplotlib.pyplot as plt
import numpy as np
import torch

from sam2roman.io.fits import FITSImage
from sam2roman.training.dataset import AstronomicalDataset
from sam2roman.training.trainer import SAM2Trainer
from sam2roman.inference.segmentor import SAM2Segmentor
from sam2roman.utils.synthetic import generate_synthetic_roman_fits
from sam2roman.visualization.display import plot_masks_overlay

# Hardware detection and A100 resource scaling
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    cuda_cap = torch.cuda.get_device_capability(0)
    num_cpus = os.cpu_count() or 4
    
    print(f"[Hardware] Accelerator: {gpu_name} ({vram_gb:.1f} GB VRAM, {num_cpus} vCPUs)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    
    if vram_gb >= 35.0:
        BATCH_SIZE = 16
        NUM_WORKERS = min(4, num_cpus)
        PIN_MEMORY = True
        EPOCHS = 10
        print(f">> [Profile: A100 40GB High-Throughput] Batch Size: {BATCH_SIZE}, Workers: {NUM_WORKERS}")
    elif vram_gb >= 20.0:
        BATCH_SIZE = 8
        NUM_WORKERS = min(2, num_cpus)
        PIN_MEMORY = True
        EPOCHS = 5
    else:
        BATCH_SIZE = 4
        NUM_WORKERS = min(2, num_cpus)
        PIN_MEMORY = True
        EPOCHS = 4
else:
    device_name = "MPS (Apple Silicon)" if torch.backends.mps.is_available() else "CPU"
    print(f"[Hardware] Device: {device_name}")
    BATCH_SIZE = 2
    NUM_WORKERS = 0
    PIN_MEMORY = False
    EPOCHS = 4

### Step 2: Prepare Master Astronomical Exposure and Ground-Truth SegMap

In [ ]:
base_dir = Path(".")
data_dir = base_dir / "sample_data"
output_dir = base_dir / "checkpoints"
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

master_fits_path = data_dir / "sample_roman_train_field.fits"
if not master_fits_path.exists():
    print(f"Generating synthetic Roman WFI field with ground truth: {master_fits_path.name}")
    image_data, header, catalog, gt_segmap = generate_synthetic_roman_fits(
        output_path=master_fits_path,
        shape=(1024, 1024),
        n_stars=60,
        n_galaxies=35,
        seed=42,
        filter_name="F158",
    )
else:
    fits_img = FITSImage.from_file(master_fits_path)
    image_data = fits_img.data
    _, _, catalog, gt_segmap = generate_synthetic_roman_fits(
        shape=(1024, 1024), n_stars=60, n_galaxies=35, seed=42
    )

print(f"Master field dimensions: {image_data.shape}")
print(f"Ground truth sources:    {len(catalog)} objects in field")

### Step 3: Create Tiled Cutouts from the Master Image

In [ ]:
full_dataset = AstronomicalDataset.from_single_fits_tiled(
    fits_image=image_data,
    segmap=gt_segmap,
    tile_size=512,
    stride=256,
    min_objects=2,
    prompt_type="point",  # Generates point prompts from foreground masks
    stretch_type="leakysb", # Default: leakySB surface brightness stretch
)

print(f"Created {len(full_dataset)} training tiles from master image!")

# Split into 80% Train, 20% Validation
n_train = max(1, int(0.8 * len(full_dataset)))
n_val = len(full_dataset) - n_train
train_ds, val_ds = torch.utils.data.random_split(
    full_dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

### Step 4: Configure `SAM2Trainer`

In [ ]:
trainer = SAM2Trainer(
    model_id="facebook/sam2.1-hiera-tiny",
    learning_rate=1e-4,
    weight_decay=1e-4,
    freeze_vision_encoder=True,
    prompt_type="point",
    use_amp=True,  # Enables bfloat16 on A100
)

print(f"Device:               {trainer.device}")
print(f"Trainable parameters: {sum(p.numel() for p in trainer.trainable_params):,}")

### Step 5: Execute Training Loop with Resource-Aware DataLoader

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

history = trainer.train(
    dataset=train_ds,
    val_dataset=val_ds,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    output_dir=output_dir,
    verbose=True,
)

if torch.cuda.is_available():
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print(f"\n[A100 Resource Stats] Peak VRAM: {peak_vram:.2f} GB / {vram_gb:.1f} GB")

### Step 6: Plot Convergence Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ep_range = range(1, EPOCHS + 1)

ax1.plot(ep_range, history["train_loss"], 'o-', label="Train Loss", color="royalblue")
if history["val_loss"]:
    ax1.plot(ep_range, history["val_loss"], 's--', label="Val Loss", color="crimson")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Combined Loss")
ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)
ax1.legend()

if history["val_iou"]:
    ax2.plot(ep_range, history["val_iou"], '^-', label="Val IoU", color="forestgreen")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Mean IoU")
    ax2.set_title("Validation IoU")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

plt.tight_layout()
plt.show()

### Step 7: Test Inference with Fine-Tuned Checkpoint

In [ ]:
best_ckpt = output_dir / "checkpoint_best.pt"
if not best_ckpt.exists():
    best_ckpt = output_dir / f"checkpoint_epoch_{EPOCHS}.pt"

val_sample = val_ds[0]
test_patch = val_sample["raw_data"]
test_pt = val_sample["point"][0].tolist()

segmentor = SAM2Segmentor(model_id="facebook/sam2.1-hiera-tiny")
segmentor.model_wrapper.model.load_state_dict(
    torch.load(best_ckpt, map_location=trainer.device)["model_state_dict"]
)
segmentor.model_wrapper.model.eval()

res = segmentor.segment_points(
    image=test_patch,
    points=[(test_pt[0], test_pt[1])],
    stretch_type="leakysb",
    verbose=True,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(val_sample["pil_image"])
ax1.plot(test_pt[0], test_pt[1], 'r*', markersize=14, label="Prompt Click")
ax1.set_title("Test Patch + Prompt")
ax1.legend()

ax2.imshow(val_sample["pil_image"])
if res.masks:
    pred_mask = res.masks[0]
    overlay = np.zeros((*pred_mask.shape, 4))
    overlay[pred_mask] = [0.2, 0.8, 0.2, 0.6]
    ax2.imshow(overlay)
    ax2.set_title(f"Fine-Tuned Mask (Area: {res.areas[0]} px, Score: {res.scores[0]:.2f})")

plt.tight_layout()
plt.show()